# Whisper Text Embedding Memmap Builder

Builds `whisper_features.dat` + `whisper_mapping.json` using the **exact same padding contract** as `memap_build.ipynb` (video) and `build_audio_embedding_memmap.py` (audio).

**Pipeline:**
1. For each game half, extract audio from the `.mkv` video file via `ffmpeg`
2. Run **OpenAI Whisper** (multilingual) → timestamped transcript segments
3. Align segments to **1 fps** frames: for frame `t`, concatenate all segment text overlapping second `[t, t+1)`
4. Embed each frame's text with **sentence-transformers** (multilingual)
5. Apply same `np.pad(mode='edge')` as video/audio: `l_pad=23`, `r_pad=22`
6. Write memmap + mapping JSON — plug-and-play with `dataset.py`

**Output shape per clip in training:** `(45, embed_dim)` — same window as video/audio.

---

### Installation (Colab)
```bash
!pip install openai-whisper sentence-transformers SoccerNet tqdm numpy
!apt-get install -y ffmpeg
```

## Cell 1 — Config (edit this)

In [ ]:
from pathlib import Path

SOCCERNET_PATH   = Path("/content/drive/MyDrive/data_sn/caption-2024")
AUDIO_ROOT       = Path("/content/drive/MyDrive/data_sn/audio_output")
HALF1_AUDIO_NAME = "1_224p_raw.wav"
HALF2_AUDIO_NAME = "2_224p_raw.wav"
OUT_FEATURE_FILE       = Path("/content/drive/MyDrive/data_sn/whisper_memmap/whisper_features.dat")
OUT_MAPPING_JSON       = Path("/content/drive/MyDrive/data_sn/whisper_memmap/whisper_mapping.json")
REFERENCE_MAPPING_JSON = Path("/content/drive/MyDrive/data_sn/video_mapping.json")

WHISPER_MODEL_SIZE = "tiny"
SBERT_MODEL_NAME   = "paraphrase-multilingual-MiniLM-L12-v2"

SPLITS           = ["train", "valid", "test"]
FRAMERATE        = 1
WINDOW_SECONDS   = 45
PAD_MODE         = "edge"
WHISPER_LANGUAGE = None
WHISPER_TASK     = "transcribe"
SBERT_BATCH_SIZE = 512
DRY_RUN          = False

WINDOW_SIZE_FRAME = FRAMERATE * WINDOW_SECONDS
L_PAD = WINDOW_SIZE_FRAME // 2 + WINDOW_SIZE_FRAME % 2
R_PAD = WINDOW_SIZE_FRAME // 2

print(f"Whisper : {WHISPER_MODEL_SIZE}  |  SBERT : {SBERT_MODEL_NAME}")
print(f"window={WINDOW_SIZE_FRAME}  l_pad={L_PAD}  r_pad={R_PAD}")
print(f"Audio root : {AUDIO_ROOT}")

## Cell 2 — Imports & model loading

In [ ]:
import json
import subprocess
import tempfile
import warnings
from pathlib import Path

import numpy as np
import torch
import whisper
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from SoccerNet.Downloader import getListGames

warnings.filterwarnings("ignore", message="FP16 is not supported on CPU")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

whisper_model = whisper.load_model(WHISPER_MODEL_SIZE, device=DEVICE)
print("Whisper ready.")

sbert_model = SentenceTransformer(SBERT_MODEL_NAME, device=DEVICE)
EMBED_DIM = sbert_model.get_sentence_embedding_dimension()
print(f"SBERT ready. embed_dim={EMBED_DIM}")

## Cell 3 — Helper functions

In [ ]:
import logging as _logging
import os
_log = _logging.getLogger("whisper_memmap")
_logging.basicConfig(level=_logging.INFO, format="%(levelname)s %(message)s")

def transcribe_audio(wav_path: Path) -> list[dict]:
    def _run(fp16):
        return whisper_model.transcribe(
            str(wav_path), language=WHISPER_LANGUAGE, task=WHISPER_TASK,
            verbose=False, fp16=fp16,
        ).get("segments", [])
    try:
        return _run(fp16=(DEVICE == "cuda"))
    except torch.cuda.OutOfMemoryError:
        _log.warning(f"Whisper GPU OOM on {wav_path.name} — retrying on CPU")
        try:
            return _run(fp16=False)
        except Exception as e:
            _log.warning(f"Whisper CPU retry failed: {e}")
            return []
    except Exception as e:
        _log.warning(f"Whisper failed on {wav_path.name}: {type(e).__name__}: {e}")
        return []

def segments_to_frame_texts(segments: list[dict], n_frames: int) -> list[str]:
    frame_texts = [""] * n_frames
    for seg in segments:
        try:
            start_s = float(seg["start"])
            end_s   = float(seg["end"])
            text    = str(seg.get("text", "")).strip()
        except (KeyError, TypeError, ValueError):
            continue
        if not text:
            continue
        for t in range(max(0, int(start_s)), min(n_frames - 1, int(end_s)) + 1):
            sep = " " if frame_texts[t] else ""
            frame_texts[t] = frame_texts[t] + sep + text
    return frame_texts

def embed_frame_texts(texts: list[str]) -> np.ndarray:
    embeddings = np.zeros((len(texts), EMBED_DIM), dtype=np.float32)
    non_empty_idx = [i for i, t in enumerate(texts) if t.strip()]
    if not non_empty_idx:
        return embeddings
    non_empty_texts = [texts[i] for i in non_empty_idx]
    def _encode(bs):
        return sbert_model.encode(
            non_empty_texts, batch_size=bs, show_progress_bar=False,
            convert_to_numpy=True, normalize_embeddings=True,
        ).astype(np.float32)
    try:
        vecs = _encode(SBERT_BATCH_SIZE)
    except torch.cuda.OutOfMemoryError:
        _log.warning("SBERT OOM — retrying batch=32")
        try:
            vecs = _encode(32)
        except Exception as e:
            _log.warning(f"SBERT failed: {e}")
            return embeddings
    except Exception as e:
        _log.warning(f"SBERT failed: {e}")
        return embeddings
    for out_i, src_i in enumerate(non_empty_idx):
        embeddings[src_i] = vecs[out_i]
    return embeddings

def align_time_axis(arr: np.ndarray, target_t: int) -> np.ndarray:
    arr = arr.astype(np.float32)
    t, d = arr.shape
    if t == target_t: return arr
    if t == 0: return np.zeros((target_t, d), dtype=np.float32)
    old_x = np.linspace(0.0, 1.0, t)
    new_x = np.linspace(0.0, 1.0, target_t)
    out = np.empty((target_t, d), dtype=np.float32)
    for j in range(d):
        out[:, j] = np.interp(new_x, old_x, arr[:, j].astype(np.float64)).astype(np.float32)
    return out

def audio_path_for_half(game: str, half_audio_name: str) -> Path:
    parts = Path(game).parts
    league, season = parts[0], parts[1]
    match = Path(*parts[2:])
    return AUDIO_ROOT / league / season / "raw" / match / half_audio_name

def process_half(game, half_audio_name, ref_n_frames=None):
    n_fallback = ref_n_frames if ref_n_frames is not None else 0
    zeros = np.zeros((n_fallback, EMBED_DIM), dtype=np.float32)

    wav_path = audio_path_for_half(game, half_audio_name)
    if not wav_path.is_file():
        _log.warning(f"Audio not found: {wav_path}")
        return zeros, True
    if wav_path.stat().st_size < 3200:
        _log.warning(f"Audio too short: {wav_path}")
        return zeros, True

    try:
        segments = transcribe_audio(wav_path)
    except Exception as e:
        _log.warning(f"Unexpected error {game}/{half_audio_name}: {e}")
        return zeros, True

    if not segments:
        n = ref_n_frames if ref_n_frames is not None else 1
        return np.zeros((n, EMBED_DIM), dtype=np.float32), False

    duration_s = max((float(s["end"]) for s in segments), default=0.0)
    n_frames   = max(int(np.ceil(duration_s)), ref_n_frames or 1)
    embeddings = embed_frame_texts(segments_to_frame_texts(segments, n_frames))
    if ref_n_frames is not None and embeddings.shape[0] != ref_n_frames:
        embeddings = align_time_axis(embeddings, ref_n_frames)
    return embeddings, False

print("Helpers ready.")

## Cell 4 — Load reference mapping & game list

In [ ]:
with open(REFERENCE_MAPPING_JSON, encoding="utf-8") as f:
    ref_mapping: dict[str, dict[str, int]] = json.load(f)

print(f"Reference mapping loaded: {len(ref_mapping)} games")

games = getListGames(SPLITS, task="caption")
print(f"Total games: {len(games)}")
print(f"First 3: {games[:3]}")

print(audio_path_for_half(games[0], HALF1_AUDIO_NAME))
print(audio_path_for_half(games[0], HALF1_AUDIO_NAME).is_file())

## Cell 5 — Pass 1: Scan all games, compute total_rows

In [ ]:
total_rows    = 0
n_zero_games  = 0
n_missing_ref = 0

for idx, game in enumerate(tqdm(games, desc="Pass 1 scan")):
    key = str(idx)
    if key not in ref_mapping:
        n_missing_ref += 1
        continue
    e = ref_mapping[key]
    ref_t1 = int(e["half1_len"]) - L_PAD - R_PAD
    ref_t2 = int(e["half2_len"]) - L_PAD - R_PAD
    a1 = audio_path_for_half(game, HALF1_AUDIO_NAME)
    a2 = audio_path_for_half(game, HALF2_AUDIO_NAME)
    if a1.is_file() and a2.is_file():
        total_rows += (ref_t1 + L_PAD + R_PAD) + (ref_t2 + L_PAD + R_PAD)
    else:
        total_rows += int(e["half1_len"]) + int(e["half2_len"])
        n_zero_games += 1

print(f"games={len(games)}  total_rows={total_rows}  embed_dim={EMBED_DIM}")
print(f"zero_fill={n_zero_games}  missing_ref={n_missing_ref}")
print(f"file size est. = {total_rows * EMBED_DIM * 4 / 1e9:.2f} GB")

## Cell 6 — Pass 2: Transcribe, embed, write memmap

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True — skipping.")
else:
    OUT_FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_FILE = OUT_FEATURE_FILE.parent / "whisper_checkpoint.json"

    # ── Resume or fresh start ─────────────────────────────────────────────────
    if CHECKPOINT_FILE.is_file() and OUT_FEATURE_FILE.is_file():
        print(f"Checkpoint found — resuming...")
        with open(CHECKPOINT_FILE, encoding="utf-8") as f:
            ckpt = json.load(f)
        cursor      = int(ckpt["cursor"])
        mapping_out = ckpt["mapping"]
        done_keys   = set(mapping_out.keys())
        mem = np.memmap(OUT_FEATURE_FILE, mode="r+", dtype=np.float32, shape=(total_rows, EMBED_DIM))
        print(f"  {len(done_keys)}/{len(games)} games done, cursor={cursor}")
    else:
        print("No checkpoint — starting fresh.")
        cursor, mapping_out, done_keys = 0, {}, set()
        mem = np.memmap(OUT_FEATURE_FILE, mode="w+", dtype=np.float32, shape=(total_rows, EMBED_DIM))

    # ── Pass 2 write ──────────────────────────────────────────────────────────
    n_zero = 0
    for idx, game in enumerate(tqdm(games, desc="Pass 2 write")):
        key = str(idx)
        if key in done_keys:
            continue
        if key not in ref_mapping:
            mapping_out[key] = {"half1_start": cursor, "half1_len": 0, "half2_start": cursor, "half2_len": 0}
            done_keys.add(key)
            continue

        e      = ref_mapping[key]
        ref_t1 = int(e["half1_len"]) - L_PAD - R_PAD
        ref_t2 = int(e["half2_len"]) - L_PAD - R_PAD

        f1, zf1 = process_half(game, HALF1_AUDIO_NAME, ref_n_frames=ref_t1)
        f2, zf2 = process_half(game, HALF2_AUDIO_NAME, ref_n_frames=ref_t2)
        if zf1 or zf2: n_zero += 1

        f1 = np.pad(f1, ((L_PAD, R_PAD), (0, 0)), mode=PAD_MODE) if not zf1 else np.zeros((int(e["half1_len"]), EMBED_DIM), dtype=np.float32)
        f2 = np.pad(f2, ((L_PAD, R_PAD), (0, 0)), mode=PAD_MODE) if not zf2 else np.zeros((int(e["half2_len"]), EMBED_DIM), dtype=np.float32)

        h1_start, h1_len = cursor, int(f1.shape[0])
        mem[h1_start:h1_start+h1_len] = f1.astype(np.float32)
        cursor += h1_len
        h2_start, h2_len = cursor, int(f2.shape[0])
        mem[h2_start:h2_start+h2_len] = f2.astype(np.float32)
        cursor += h2_len

        mapping_out[key] = {"half1_start": h1_start, "half1_len": h1_len, "half2_start": h2_start, "half2_len": h2_len}
        done_keys.add(key)

        # Save checkpoint after every game
        mem.flush()
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump({"cursor": cursor, "mapping": mapping_out}, f)

    # ── Finalise ──────────────────────────────────────────────────────────────
    mem.flush()
    del mem
    with open(OUT_MAPPING_JSON, "w", encoding="utf-8") as f:
        json.dump(mapping_out, f, indent=2)
    if cursor == total_rows:
        CHECKPOINT_FILE.unlink(missing_ok=True)
        print("Complete — checkpoint deleted.")
    print(f"\nWrote : {OUT_FEATURE_FILE}")
    print(f"Wrote : {OUT_MAPPING_JSON}")
    print(f"zero_filled={n_zero}  cursor={cursor}/{total_rows}")

## Cell 7 — Verification

In [ ]:
import json
import numpy as np
import random

with open(OUT_MAPPING_JSON, encoding="utf-8") as f:
    wmap = json.load(f)
with open(REFERENCE_MAPPING_JSON, encoding="utf-8") as f:
    vmap = json.load(f)

max_row = max(e["half2_start"] + e["half2_len"] for e in wmap.values() if e["half2_len"] > 0)
mem_check = np.memmap(OUT_FEATURE_FILE, mode="r", dtype=np.float32, shape=(max_row, EMBED_DIM))
print(f"Memmap shape : {mem_check.shape}")
print(f"Games in map : {len(wmap)}")

mismatches = []
for key in wmap:
    if key not in vmap:
        continue
    we, ve = wmap[key], vmap[key]
    if we["half1_len"] != ve["half1_len"] or we["half2_len"] != ve["half2_len"]:
        mismatches.append((key, we, ve))

if mismatches:
    print(f"WARNING: {len(mismatches)} half-length mismatches vs video mapping:")
    for key, we, ve in mismatches[:5]:
        print(f"  game {key}: whisper h1={we['half1_len']} vs video h1={ve['half1_len']}, h2={we['half2_len']} vs video h2={ve['half2_len']}")
else:
    print("All half lengths match video mapping.")

sample_key = random.choice(list(wmap.keys()))
entry = wmap[sample_key]
clip  = np.array(mem_check[entry["half1_start"] : entry["half1_start"] + WINDOW_SIZE_FRAME])
print(f"\nSample clip : game={sample_key} shape={clip.shape}")
print(f"  non-zero rows : {(np.abs(clip) > 0).any(axis=1).sum()} / {clip.shape[0]}")
print(f"  norm[0]       : {np.linalg.norm(clip[0]):.4f}")

del mem_check
print("Verification complete.")

## Cell 8 — Usage in training (copy into captioning.py / contrastive_learning.py)

The whisper memmap uses the **exact same interface** as audio.
In `captioning.py`, add a `--whisper_dir` flag and pass it like `--master_audio_dir`:

```python
# ── In argument parser ───────────────────────────────────────────────────────
parser.add_argument("--whisper_dir", type=str, default=None,
                    help="Dir containing whisper_features.dat + whisper_mapping.json")

# ── In dataset construction ──────────────────────────────────────────────────
whisper_dat  = Path(args.whisper_dir) / "whisper_features.dat"   if args.whisper_dir else None
whisper_json = Path(args.whisper_dir) / "whisper_mapping.json"  if args.whisper_dir else None

# Load memmap (same as audio)
if whisper_dat and whisper_dat.is_file():
    with open(whisper_json) as f:
        wmap = json.load(f)
    wrows = max(e['half2_start'] + e['half2_len'] for e in wmap.values())
    wdim  = 384  # or 768 depending on SBERT_MODEL_NAME
    whisper_memmap = np.memmap(whisper_dat, mode='r', dtype='float32', shape=(wrows, wdim))
```

Then use `whisper_memmap` + `wmap` exactly like `audio_memmap` + `audio_mapping` in `dataset.py`.
Clip extraction is identical — frame index → `memmap[start : start + 45]` → `(45, embed_dim)` tensor.